# MMALS RC2O-prep — Selector-Regret and Feature-Only Regime Audit

**Purpose:** prepare the transition from RC2N threshold tuning to RC2O regime-certified deployment.

This notebook is not a new MMALS training harness. It is an audit harness.

It reads completed RC2N-d ZIP packages and produces:

- `selector_regret_audit.csv`
- `feature_only_regime_audit.csv`
- `seed_specimen_audit.csv`
- `false_lock_false_release_table.csv`
- `regime_feature_summary.csv`
- `regime_confusion_matrix.csv`
- `historical_lineage.csv`
- `rc2o_prep_summary.pdf`

Core claim:

> RC2N-d is a controlled negative result. It nearly closes MNIST clean-regime regret, but it false-releases no-repair on FashionMNIST seed 2. This closes the entangled threshold-controller chapter and motivates RC2O: feature-only regime estimation plus regime-conditional / conformal safe deployment.


## Historical note — why this artifact exists

The path matters as much as the final score.

- **RC1b** established the guarded context-gap core.
- **RC2M-alpha/beta** showed dynamic boundary diagnostics were informative but unsafe alone.
- **RC2N** introduced safe regime-state deployment.
- **RC2N-b/c** introduced clean release and no-repair release.
- **RC2N-d** relaxed no-repair release and produced the decisive controlled pair:

| Version | FashionMNIST | MNIST |
|---|---|---|
| RC2N-c | safe | too conservative |
| RC2N-d | false release on seed 2 | near no-repair oracle |

This is not random iteration. It is the research program discovering its next object:

> Regime detection and policy selection must be decoupled.


## 12-year-old version

Imagine a mushroom network under a forest.

Sometimes the forest is clear. The mushroom can see the right tree. That is like MNIST. The simple path can be best.

Sometimes the forest is foggy. Several trees look similar. That is like FashionMNIST. The mushroom should keep the guarded path.

RC2N-c was careful: it protected the foggy forest but stayed too guarded in the clear forest.

RC2N-d was braver: it helped the clear forest, but once it trusted the simple path in the foggy forest.

So the next rule is:

1. First ask: is the forest clear, foggy, weak, or unknown?
2. Only after that, choose the path.
3. If the forest is unknown, keep the safe path or abstain.

That is RC2O.


In [ ]:
from pathlib import Path
import zipfile, json, re, math
import pandas as pd
import numpy as np

OUT_DIR = Path("./rc2o_prep_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Update these paths in Colab/Drive if needed.
PACKAGE_PATHS = [
    Path("/content/drive/MyDrive/MMALS/v1_1_rc2m_alpha_boundary_first_frozen_selector/RC2N_D_RELAXED_CLEAN_NO_REPAIR_RELEASE_GATE_HYDRO_AUDIT/MMALS_v11_RC2N_D_RELAXED_CLEAN_NO_REPAIR_RELEASE_GATE_HYDRO_AUDIT_MNIST_robust_20260605T161931Z_seeds5_package.zip"),
    Path("/content/drive/MyDrive/MMALS/v1_1_rc2m_alpha_boundary_first_frozen_selector/RC2N_D_RELAXED_CLEAN_NO_REPAIR_RELEASE_GATE_HYDRO_AUDIT/MMALS_v11_RC2N_D_RELAXED_CLEAN_NO_REPAIR_RELEASE_GATE_HYDRO_AUDIT_FashionMNIST_robust_20260605T181556Z_seeds5_package.zip"),
]

# Local fallback when run from the generated package.
LOCAL_FALLBACKS = [
    Path("/mnt/data/MMALS_v11_RC2N_D_RELAXED_CLEAN_NO_REPAIR_RELEASE_GATE_HYDRO_AUDIT_MNIST_robust_20260605T161931Z_seeds5_package.zip"),
    Path("/mnt/data/MMALS_v11_RC2N_D_RELAXED_CLEAN_NO_REPAIR_RELEASE_GATE_HYDRO_AUDIT_FashionMNIST_robust_20260605T181556Z_seeds5_package.zip"),
]

PACKAGE_PATHS = [p for p in PACKAGE_PATHS if p.exists()] or [p for p in LOCAL_FALLBACKS if p.exists()]
print("Packages found:", len(PACKAGE_PATHS))
for p in PACKAGE_PATHS:
    print("-", p)


## Audit implementation

The GitHub package includes already generated CSV/PDF outputs.  
For reproducibility, this notebook also includes the full audit implementation in the accompanying script:

```text
src/rc2o_prep_audit.py
```

The script:
1. extracts robust ZIP packages,
2. reads final scorecards and policy traces,
3. computes selector regret,
4. computes feature-only regime labels without dataset name,
5. highlights MNIST seed 1 and FashionMNIST seed 2,
6. exports CSV/PDF audit artifacts.


In [ ]:
# Load generated outputs if present.
from pathlib import Path
import pandas as pd

OUT_DIR = Path("./rc2o_prep_outputs")
if not OUT_DIR.exists():
    OUT_DIR = Path("/mnt/data/rc2o_prep_outputs")

selector_regret_audit = pd.read_csv(OUT_DIR / "selector_regret_audit.csv")
feature_only_regime_audit = pd.read_csv(OUT_DIR / "feature_only_regime_audit.csv")
seed_specimen_audit = pd.read_csv(OUT_DIR / "seed_specimen_audit.csv")
false_lock_false_release_table = pd.read_csv(OUT_DIR / "false_lock_false_release_table.csv")

print("Selector-regret rows:", len(selector_regret_audit))
print("Feature-only regime rows:", len(feature_only_regime_audit))
display(seed_specimen_audit)
display(false_lock_false_release_table)


## Interpretation rule

The first RC2O-prep estimator deliberately keeps three things separate:

| Field | Meaning |
|---|---|
| `dataset` | only identity; forbidden for feature-only regime inference |
| `empirical_regime_label_weak` | human weak label used for analysis |
| `feature_only_regime_label` | label inferred from validation/audit-memory global signals only |

This avoids the dataset-recognizer trap.

RC2O must not learn:

```text
MNIST -> clean
FashionMNIST -> ambiguous
```

It must learn:

```text
validation-memory trace -> clean / ambiguous / unknown
```


## Next research object

**RC2O — Decoupled Regime Estimator + Conformal Safe Selector**

Planned structure:

```text
Validation/audit-memory traces
        ↓
Feature-only regime estimator
        ↓
Regime confidence set: {clean}, {ambiguous}, {clean, ambiguous}, {unknown}
        ↓
Regime-conditional policy selector
        ↓
release / lock / fallback / abstain
```

The key change:

> Regime is no longer a soft feature inside the release gate. Certified regime becomes a deployment constraint.
